# Cold Start Strategy Evaluation

## Purpose
Evaluate and compare different cold start strategies before any model training.
The goal is to assess diversity, representativeness, and informativeness of
initial labeled samples.

## Strategies Tested
- Random
- Simple diversity (image statistics)
- Deep feature diversity
- Entropy-based uncertainty
- Weak supervision
- Self-supervised features

## Outputs
- Selected sample indices
- Diversity statistics
- Feature-space coverage plots


In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import torch

sys.path.append(str(Path("..").resolve()))

from src.cold_start_strategies import ColdStartStrategies
from src.data_modules.factory import load_dataset
from config.config import ActiveLearningConfig


In [ ]:
config = ActiveLearningConfig.from_yaml("../configs/example_segmentation.yaml")
dataset = load_dataset(config, split="train")

all_indices = list(range(len(dataset)))
n_samples = int(0.05 * len(dataset))  # e.g. 5%

In [ ]:
cold_start = ColdStartStrategies(dataset, config)

strategies = [
    "random",
    "simple_diversity",
    "diversity",
    "entropy_based_uncertainty",
    "weak_supervision",
    "self_supervised",
]

selections = {}

for s in strategies:
    selected = cold_start.apply(s, n_samples, all_indices)
    selections[s] = selected
    print(f"{s}: selected {len(selected)} samples")


In [ ]:
def compute_diversity(features):
    dists = []
    for i in range(len(features)):
        for j in range(i+1, len(features)):
            dists.append(np.linalg.norm(features[i] - features[j]))
    return np.mean(dists)

def label_histogram(indices):
    labels = []
    for i in indices:
        _, mask = dataset[i]
        labels.append(mask.unique().cpu().numpy())
    return np.unique(np.concatenate(labels), return_counts=True)


In [ ]:
features_all = cold_start._extract_features(all_indices)

features_selected = {
    s: features_all[[all_indices.index(i) for i in idxs]]
    for s, idxs in selections.items()
}

In [ ]:
import pandas as pd

rows = []
for s, feats in features_selected.items():
    rows.append({
        "strategy": s,
        "diversity": compute_diversity(feats),
        "coverage": compute_diversity(feats) / compute_diversity(features_all),
    })

df = pd.DataFrame(rows)
df.sort_values("coverage", ascending=False)


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
proj_all = pca.fit_transform(features_all)

plt.figure(figsize=(6,6))
plt.scatter(proj_all[:,0], proj_all[:,1], s=2, alpha=0.3, label="All")

for s, feats in features_selected.items():
    proj_sel = pca.transform(feats)
    plt.scatter(proj_sel[:,0], proj_sel[:,1], label=s, s=30)

plt.legend()
plt.title("Cold Start Coverage in Feature Space")
plt.show()
